### **Carga local de Indicadores_vano_clima**

Este cuaderno parte del archivo local `data/Indicadores_vano_v1.csv`, normaliza `FECHA` al formato `%Y-%m-%d %H:%M:%S` y usa Open-Meteo para completar únicamente los valores climáticos faltantes.


In [1]:
from pathlib import Path
import pandas as pd

FECHA_FORMAT = "%Y-%m-%d %H:%M:%S"
def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate
    return cwd


PROJECT_ROOT = resolve_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v1.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v2.csv"

Xdata = pd.read_csv(INPUT_PATH)
Xdata.index.name = "original_index"



In [2]:
Xdata

,CIRCUITO,FID_SW,COD_EQ_PROTEGE,FID_VANO,T_USUS_EQ_PROT,LVSW,CNT_VN,CNT_VN_SW,FECHA,DURACION,...,clouds_15,clouds_16,clouds_17,clouds_18,clouds_19,clouds_20,clouds_21,clouds_22,clouds_23,clouds_24
original_index,,,,,,,,,,,,,,,,,,,,,
0,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-03 23:56:04,0.008,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-18 17:08:14,0.008,...,100.0,100.0,84.0,99.0,99.0,98.0,98.0,97.0,98.0,100.0
2,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-19 20:25:50,0.008,...,79.0,99.0,100.0,100.0,100.0,97.0,100.0,100.0,99.0,98.0
3,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-10 21:33:41,0.008,...,69.0,93.0,100.0,55.0,61.0,32.0,56.0,59.0,58.0,76.0
4,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-19 15:59:10,0.041,...,99.0,97.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159465,GTO23L13,31291927,L22261,357636441,15,1.657,17,21,2025-11-16 06:55:00,8.833,...,100.0,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0
159466,GTO23L13,31291927,L22261,357636443,15,1.792,18,21,2025-11-16 06:55:00,8.833,...,100.0,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0
159467,GTO23L13,31291927,L22261,357636451,15,1.951,19,21,2025-11-16 06:55:00,8.833,...,100.0,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0


## **Normalizar FECHA y convertir a UTC**


In [3]:
import os
import time
import numpy as np
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

REQUIRED_COLUMNS = ["FECHA", "X1", "Y1", "X2", "Y2"]
missing_required = [column for column in REQUIRED_COLUMNS if column not in Xdata.columns]
if missing_required:
    raise ValueError(f"Faltan columnas requeridas: {missing_required}")

def parse_local_bogota_to_utc(series_str: pd.Series) -> pd.Series:
    dt_local = pd.to_datetime(series_str, errors="coerce", format=FECHA_FORMAT)
    dt_local = dt_local.dt.tz_localize(ZoneInfo("America/Bogota"), nonexistent="NaT", ambiguous="NaT")
    return dt_local.dt.tz_convert(timezone.utc)

print("Normalizando FECHA...")
Xdata["FECHA"] = pd.to_datetime(Xdata["FECHA"], errors="coerce").dt.strftime(FECHA_FORMAT)
dt_utc = parse_local_bogota_to_utc(Xdata["FECHA"].astype(str))

DF_ISO = Xdata.copy()
DF_ISO["event_time_iso_utc"] = dt_utc

invalid_dates = DF_ISO["event_time_iso_utc"].isna().sum()
print(f"Filas totales: {len(DF_ISO):,}")
print(f"Fechas inválidas: {invalid_dates:,}")

if invalid_dates:
    raise ValueError("Hay fechas inválidas en FECHA; revisa esas filas antes de llamar la API.")

DF_ISO[["FECHA", "event_time_iso_utc", "X1", "Y1", "X2", "Y2"]].head()


Normalizando FECHA...
Filas totales: 159,470
Fechas inválidas: 0


,FECHA,event_time_iso_utc,X1,Y1,X2,Y2
original_index,,,,,,
0,2025-11-03 23:56:04,2025-11-04 04:56:04+00:00,-75.452164,5.627635,-75.452412,5.626978
1,2025-11-18 17:08:14,2025-11-18 22:08:14+00:00,-75.452164,5.627635,-75.452412,5.626978
2,2025-11-19 20:25:50,2025-11-20 01:25:50+00:00,-75.452164,5.627635,-75.452412,5.626978
3,2025-12-10 21:33:41,2025-12-11 02:33:41+00:00,-75.452164,5.627635,-75.452412,5.626978
4,2025-12-19 15:59:10,2025-12-19 20:59:10+00:00,-75.452164,5.627635,-75.452412,5.626978


## **Diagnóstico de columnas climáticas faltantes**


In [4]:
DF_ISO[["FECHA", "event_time_iso_utc", "X1", "Y1", "X2", "Y2"]].head()


,FECHA,event_time_iso_utc,X1,Y1,X2,Y2
original_index,,,,,,
0,2025-11-03 23:56:04,2025-11-04 04:56:04+00:00,-75.452164,5.627635,-75.452412,5.626978
1,2025-11-18 17:08:14,2025-11-18 22:08:14+00:00,-75.452164,5.627635,-75.452412,5.626978
2,2025-11-19 20:25:50,2025-11-20 01:25:50+00:00,-75.452164,5.627635,-75.452412,5.626978
3,2025-12-10 21:33:41,2025-12-11 02:33:41+00:00,-75.452164,5.627635,-75.452412,5.626978
4,2025-12-19 15:59:10,2025-12-19 20:59:10+00:00,-75.452164,5.627635,-75.452412,5.626978


## **Filas que necesitan completar clima**


In [5]:
LAGS = 25  # t, t-1, ..., t-24

VAR_MAP = {
    "prep": "precipitation",
    "pres": "pressure_msl",
    "sp": "surface_pressure",
    "rh": "relative_humidity_2m",
    "solar_rad": "shortwave_radiation",
    "temp": "temperature_2m",
    "wind_gust_spd": "wind_gusts_10m",
    "wind_spd": "wind_speed_10m",
    "clouds": "cloud_cover",
}
VARS_SHORT = list(VAR_MAP.keys())
CLIMATE_COLUMNS = [f"{short}_{lag}" for short in VARS_SHORT for lag in range(LAGS)]

missing_climate_columns = [column for column in CLIMATE_COLUMNS if column not in DF_ISO.columns]
if missing_climate_columns:
    print(f"Se crearán {len(missing_climate_columns)} columnas climáticas que no existen en el archivo.")
    for column in missing_climate_columns:
        DF_ISO[column] = np.nan

rows_with_missing_climate = DF_ISO[CLIMATE_COLUMNS].isna().any(axis=1)
pending_index = DF_ISO.index[rows_with_missing_climate].tolist()

print(f"Columnas climáticas esperadas: {len(CLIMATE_COLUMNS):,}")
print(f"Filas con algún clima faltante: {len(pending_index):,}")
DF_ISO.loc[pending_index, ["FECHA", "X1", "Y1", "X2", "Y2"] + CLIMATE_COLUMNS[:5]].head()


Columnas climáticas esperadas: 225
Filas con algún clima faltante: 0


,FECHA,X1,Y1,X2,Y2,prep_0,prep_1,prep_2,prep_3,prep_4
original_index,,,,,,,,,,


## **Completar datos climáticos faltantes con Open-Meteo**


In [6]:
ALL_API_VARS = [VAR_MAP[k] for k in VAR_MAP]
ARCHIVE_SAFE_API_VARS = [v for v in ALL_API_VARS]


### **Helpers**

In [7]:
# ==== Helpers ====
def to_utc_iso(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).replace(tzinfo=timezone.utc).isoformat().replace("+00:00","Z")

def hour_floor(dt: datetime) -> datetime:
    return dt.replace(minute=0, second=0, microsecond=0, tzinfo=timezone.utc)

def _call_openmeteo(url: str, params: dict, var_order: list[str]) -> dict:
    req = {"latitude": params["latitude"], "longitude": params["longitude"], "timezone": "UTC", "hourly": var_order}
    if url == FORECAST_URL:
        if "past_days" in params:
            req["past_days"] = params["past_days"]
    else:
        req["start_date"] = params["start_date"]
        req["end_date"] = params["end_date"]
    responses = openmeteo.weather_api(url, params=req)
    resp = responses[0]
    hourly = resp.Hourly()

    # Create a Timezone-Aware Pandas Index immediately
    times = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    out = {"time": times}
    for i, var_name in enumerate(var_order):
        out[var_name] = hourly.Variables(i).ValuesAsNumpy()
    return out

def _slice_time_window(payload: dict, start_dt: datetime, end_dt: datetime) -> dict:
    # We force 't' to be a Pandas DatetimeIndex (UTC) so it can compare with start_dt (UTC)
    t = pd.to_datetime(payload["time"], utc=True)

    # Now both sides are UTC-aware. No more Error.
    mask = (t >= start_dt) & (t <= end_dt)

    sliced = {"time": t[mask]}
    for k, v in payload.items():
        if k == "time": continue
        # Only slice if the length matches (safety check)
        if len(v) == len(t):
            sliced[k] = v[mask]
        else:
            sliced[k] = v
    return sliced

def _combine_payloads(p_list: list[dict]) -> dict:
    all_times = pd.Index([], dtype='datetime64[ns, UTC]') # Force UTC Index
    for p in p_list:
        all_times = all_times.union(pd.Index(p["time"]))
    all_times = all_times.sort_values()

    # Validated: all_times is now a UTC Pandas Index
    out = {"time": all_times}

    keys = set().union(*[set(p.keys()) for p in p_list]) - {"time"}
    for k in keys:
        series = pd.Series(index=all_times, dtype=float)
        for p in p_list:
            if k in p:
                # Make sure the source index is also compatible
                src_idx = pd.Index(p["time"])
                s = pd.Series(p[k], index=src_idx)
                series.update(s)
        out[k] = series.reindex(all_times).to_numpy()
    return out

def get_hourly_window(lat: float, lon: float, start_dt_utc: datetime, end_dt_utc: datetime) -> dict:
    now_utc = datetime.now(timezone.utc)
    boundary = now_utc - timedelta(days=5)
    payloads = []
    if end_dt_utc <= boundary:
        params = {"latitude": lat, "longitude": lon, "start_date": start_dt_utc.date().isoformat(), "end_date": end_dt_utc.date().isoformat()}
        payloads.append(_call_openmeteo(ARCHIVE_URL, params, ARCHIVE_SAFE_API_VARS))
    elif start_dt_utc >= boundary:
        diff_days = (now_utc - start_dt_utc).days + 1
        params = {"latitude": lat, "longitude": lon, "past_days": min(7, max(1, diff_days))}
        payloads.append(_call_openmeteo(FORECAST_URL, params, ALL_API_VARS))
    else:
        params_a = {"latitude": lat, "longitude": lon, "start_date": start_dt_utc.date().isoformat(), "end_date": boundary.date().isoformat()}
        pa = _call_openmeteo(ARCHIVE_URL, params_a, ARCHIVE_SAFE_API_VARS)
        diff_days = (now_utc - boundary).days + 1
        params_f = {"latitude": lat, "longitude": lon, "past_days": min(7, max(1, diff_days))}
        pf = _call_openmeteo(FORECAST_URL, params_f, ALL_API_VARS)
        payloads.extend([pa, pf])
    combined = _combine_payloads(payloads)
    return _slice_time_window(combined, start_dt_utc, end_dt_utc)

## **Configuración de API y archivo de salida**


In [8]:
import requests_cache
from retry_requests import retry
import openmeteo_requests
from tqdm.auto import tqdm

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
ARCHIVE_URL  = "https://archive-api.open-meteo.com/v1/archive"

SAVE_EVERY_ROWS = 100
REQUEST_SLEEP_SECONDS = 0.35
MAX_ROWS_TO_PROCESS = None  # Usa un entero para probar con pocas filas, por ejemplo 10.

cache_session = requests_cache.CachedSession(str(PROJECT_ROOT / "data" / ".openmeteo_cache"), expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

STOP_FILE = PROJECT_ROOT / "data" / "STOP_OPENMETEO_LIMIT.txt"
if STOP_FILE.exists():
    raise RuntimeError(
        f"Se encontró {STOP_FILE}. Ya se detectó un límite en una ejecución previa. "
        "Borra ese archivo si deseas reintentar."
    )

if OUTPUT_PATH.exists():
    print(f"Se encontró un archivo de salida previo. Se continuará desde: {OUTPUT_PATH}")
    DF_FILLED = pd.read_csv(OUTPUT_PATH)
    DF_FILLED["FECHA"] = pd.to_datetime(DF_FILLED["FECHA"], errors="coerce").dt.strftime(FECHA_FORMAT)
    DF_FILLED["event_time_iso_utc"] = parse_local_bogota_to_utc(DF_FILLED["FECHA"].astype(str))
    for column in CLIMATE_COLUMNS:
        if column not in DF_FILLED.columns:
            DF_FILLED[column] = np.nan
else:
    DF_FILLED = DF_ISO.copy()

pending_mask = DF_FILLED[CLIMATE_COLUMNS].isna().any(axis=1)
pending_index = DF_FILLED.index[pending_mask].tolist()
if MAX_ROWS_TO_PROCESS is not None:
    pending_index = pending_index[:MAX_ROWS_TO_PROCESS]

print(f"Archivo de entrada: {INPUT_PATH}")
print(f"Archivo de salida: {OUTPUT_PATH}")
print(f"Filas pendientes a consultar en API: {len(pending_index):,}")


Archivo de entrada: /Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/data/Indicadores_vano_v1.csv
Archivo de salida: /Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/data/Indicadores_vano_v2.csv
Filas pendientes a consultar en API: 0


In [9]:
DF_FILLED.loc[pending_index, ["FECHA", "X1", "Y1", "X2", "Y2"] + CLIMATE_COLUMNS[:5]].head()


,FECHA,X1,Y1,X2,Y2,prep_0,prep_1,prep_2,prep_3,prep_4
original_index,,,,,,,,,,


In [10]:
def save_completed_dataset(df: pd.DataFrame, output_path: Path) -> None:
    columns_to_drop = [column for column in ["event_time_iso_utc"] if column in df.columns]
    df.drop(columns=columns_to_drop).to_csv(output_path, index=False)

processed_rows = 0
stop_all = False

for idx in tqdm(pending_index, desc="Completando clima", unit="fila"):
    lon = pd.to_numeric(DF_FILLED.at[idx, "X1"], errors="coerce")
    lat = pd.to_numeric(DF_FILLED.at[idx, "Y1"], errors="coerce")
    event_utc = pd.to_datetime(DF_FILLED.at[idx, "event_time_iso_utc"], utc=True, errors="coerce")

    if pd.isna(event_utc) or pd.isna(lat) or pd.isna(lon):
        continue

    try:
        event_hour_utc = hour_floor(event_utc.to_pydatetime())
        start_dt = event_hour_utc - timedelta(hours=24)
        end_dt = event_hour_utc
        payload = get_hourly_window(float(lat), float(lon), start_dt, end_dt)
    except Exception as exc:
        msg = str(exc)
        is_limit = (
            "Daily API request limit exceeded" in msg
            or "Hourly API request limit exceeded" in msg
            or "rate limit" in msg.lower()
            or "too many requests" in msg.lower()
            or "429" in msg
        )
        if is_limit:
            STOP_FILE.write_text(f"Open-Meteo limit detected | index={idx}\n{msg}\n")
            print(f"Límite de API detectado en índice {idx}. Se guarda avance parcial y se detiene.")
            stop_all = True
            break

        print(f"Error en índice {idx}: {exc}")
        continue

    desired_times = [event_hour_utc - timedelta(hours=lag) for lag in range(LAGS)]
    desired_iso = [to_utc_iso(t) for t in desired_times]
    time_index = [to_utc_iso(pd.Timestamp(t).to_pydatetime()) for t in payload["time"]]
    pos = {ts: position for position, ts in enumerate(time_index)}

    any_filled = False
    for short_name, api_name in VAR_MAP.items():
        series_vals = payload.get(api_name)
        if series_vals is None:
            continue

        for lag, ts in enumerate(desired_iso):
            column = f"{short_name}_{lag}"
            if not pd.isna(DF_FILLED.at[idx, column]):
                continue

            position = pos.get(ts)
            if position is not None:
                DF_FILLED.at[idx, column] = series_vals[position]
                any_filled = True

    if any_filled:
        processed_rows += 1

    if processed_rows and processed_rows % SAVE_EVERY_ROWS == 0:
        save_completed_dataset(DF_FILLED, OUTPUT_PATH)
        print(f"Avance guardado: {processed_rows:,} filas completadas en esta ejecución.")

    time.sleep(REQUEST_SLEEP_SECONDS)

save_completed_dataset(DF_FILLED, OUTPUT_PATH)

remaining_rows = DF_FILLED[CLIMATE_COLUMNS].isna().any(axis=1).sum()
print(f"Archivo guardado en: {OUTPUT_PATH}")
print(f"Filas completadas en esta ejecución: {processed_rows:,}")
print(f"Filas con clima pendiente después de guardar: {remaining_rows:,}")

if stop_all:
    print("Ejecución detenida por límite de API. Puedes reintentar después de borrar STOP_OPENMETEO_LIMIT.txt.")


Completando clima: 0fila [00:00, ?fila/s]

Archivo guardado en: /Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/data/Indicadores_vano_v2.csv
Filas completadas en esta ejecución: 0
Filas con clima pendiente después de guardar: 0


In [11]:
# Reporte de vacíos después de completar con API, antes de la limpieza final.
API_REPORT_SOURCE = DF_FILLED.drop(columns=["event_time_iso_utc"], errors="ignore")
api_rows = len(API_REPORT_SOURCE)
api_missing = API_REPORT_SOURCE.isna().sum()
reporte_vacios_api = (
    pd.DataFrame({
        "columna": api_missing.index,
        "vacios_api": api_missing.values,
        "porcentaje_vacios_api": api_missing.values / api_rows * 100,
    })
    .query("vacios_api > 0")
    .sort_values(["porcentaje_vacios_api", "vacios_api", "columna"], ascending=[False, False, True])
    .reset_index(drop=True)
)

print("Reporte después de API, antes de limpieza final")
print(f"Filas evaluadas: {api_rows:,}")
print(f"Columnas con vacíos > 0: {len(reporte_vacios_api):,}")
display(reporte_vacios_api)

# Limpieza final y generación del dataset v2.
# OUTPUT_PATH conserva el resultado completado por API.
# FINAL_OUTPUT_PATH guarda el resultado final después de imputar vacíos y eliminar ALTURA vacía.
FINAL_OUTPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v3.csv"
NUMERIC_ZERO_FILL_COLUMNS = [
    "PROMEDIO_KWH_VANO",
    "PROMEDIO_KWH_TRF",
    "NR_T",
    "CNT_USUS",
    "CAPACIDAD_NOMINAL",
]
TEXT_ZERO_FILL_COLUMNS = ["CALIBRE_NEUTRO"]
ZERO_FILL_COLUMNS = NUMERIC_ZERO_FILL_COLUMNS + TEXT_ZERO_FILL_COLUMNS
FECHA_TRF_COLUMN = "FECHA_OPERACION_TRF"
FECHA_VANO_COLUMN = "FECHA_OPERACION_VANO"
ALTURA_COLUMN = "ALTURA"

required_final_columns = ZERO_FILL_COLUMNS + [FECHA_TRF_COLUMN, FECHA_VANO_COLUMN, ALTURA_COLUMN]
missing_final_columns = [column for column in required_final_columns if column not in DF_FILLED.columns]
if missing_final_columns:
    raise ValueError(f"Faltan columnas requeridas para la limpieza final: {missing_final_columns}")

DF_CLEAN = DF_FILLED.copy()

zero_fill_summary = {}
for column in NUMERIC_ZERO_FILL_COLUMNS:
    original_series = DF_CLEAN[column]
    blank_mask = original_series.isna() | original_series.astype(str).str.strip().eq("")
    zero_fill_summary[column] = int(blank_mask.sum())
    DF_CLEAN[column] = pd.to_numeric(original_series, errors="coerce").fillna(0)

for column in TEXT_ZERO_FILL_COLUMNS:
    blank_mask = DF_CLEAN[column].isna() | DF_CLEAN[column].astype(str).str.strip().eq("")
    zero_fill_summary[column] = int(blank_mask.sum())
    DF_CLEAN[column] = DF_CLEAN[column].mask(blank_mask, "0")

fecha_trf_blank_mask = DF_CLEAN[FECHA_TRF_COLUMN].isna() | DF_CLEAN[FECHA_TRF_COLUMN].astype(str).str.strip().eq("")
fecha_trf_replacements = int(fecha_trf_blank_mask.sum())
DF_CLEAN.loc[fecha_trf_blank_mask, FECHA_TRF_COLUMN] = DF_CLEAN.loc[fecha_trf_blank_mask, FECHA_VANO_COLUMN]

rows_before_altura = len(DF_CLEAN)
DF_CLEAN = DF_CLEAN.dropna(subset=[ALTURA_COLUMN]).copy()
rows_removed_altura = rows_before_altura - len(DF_CLEAN)

# Guardar el resultado final v3. No se sobreescribe OUTPUT_PATH.
save_completed_dataset(DF_CLEAN, FINAL_OUTPUT_PATH)

# Reporte de vacíos después de limpieza final: solo se muestra en el notebook.
FINAL_REPORT_SOURCE = DF_CLEAN.drop(columns=["event_time_iso_utc"], errors="ignore")
final_rows = len(FINAL_REPORT_SOURCE)
final_missing = FINAL_REPORT_SOURCE.isna().sum()
reporte_vacios_final = (
    pd.DataFrame({
        "columna": final_missing.index,
        "vacios_final": final_missing.values,
        "porcentaje_vacios_final": final_missing.values / final_rows * 100,
    })
    .query("vacios_final > 0")
    .sort_values(["porcentaje_vacios_final", "vacios_final", "columna"], ascending=[False, False, True])
    .reset_index(drop=True)
)




Reporte después de API, antes de limpieza final
Filas evaluadas: 159,470
Columnas con vacíos > 0: 4


,columna,vacios_api,porcentaje_vacios_api
0,CODIGO,122024,76.518467
1,FID_TRAFO,122024,76.518467
2,LONG_CRUCETA,19002,11.915721
3,NORMA,8509,5.335800


In [ ]:
reporte_vacios_final

,columna,vacios_final,porcentaje_vacios_final
0,CODIGO,122024,76.518467
1,FID_TRAFO,122024,76.518467
2,LONG_CRUCETA,19002,11.915721
3,NORMA,8509,5.335800


In [ ]:
DF_CLEAN

,CIRCUITO,FID_SW,COD_EQ_PROTEGE,FID_VANO,T_USUS_EQ_PROT,LVSW,CNT_VN,CNT_VN_SW,FECHA,DURACION,...,clouds_16,clouds_17,clouds_18,clouds_19,clouds_20,clouds_21,clouds_22,clouds_23,clouds_24,event_time_iso_utc
original_index,,,,,,,,,,,,,,,,,,,,,
0,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-03 23:56:04,0.008,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,2025-11-04 04:56:04+00:00
1,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-18 17:08:14,0.008,...,100.0,84.0,99.0,99.0,98.0,98.0,97.0,98.0,100.0,2025-11-18 22:08:14+00:00
2,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-19 20:25:50,0.008,...,99.0,100.0,100.0,100.0,97.0,100.0,100.0,99.0,98.0,2025-11-20 01:25:50+00:00
3,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-10 21:33:41,0.008,...,93.0,100.0,55.0,61.0,32.0,56.0,59.0,58.0,76.0,2025-12-11 02:33:41+00:00
4,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-19 15:59:10,0.041,...,97.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,2025-12-19 20:59:10+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159465,GTO23L13,31291927,L22261,357636441,15,1.657,17,21,2025-11-16 06:55:00,8.833,...,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0,2025-11-16 11:55:00+00:00
159466,GTO23L13,31291927,L22261,357636443,15,1.792,18,21,2025-11-16 06:55:00,8.833,...,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0,2025-11-16 11:55:00+00:00
159467,GTO23L13,31291927,L22261,357636451,15,1.951,19,21,2025-11-16 06:55:00,8.833,...,100.0,88.0,61.0,98.0,55.0,68.0,87.0,100.0,100.0,2025-11-16 11:55:00+00:00
